# 04 — Feature Engineering & Time-Series Preparation

**Objective:** Transform the cleaned Delhi AQI dataset into a training-ready, chronologically-split feature set (PRD Milestone 3 / ML-FE-001..004, ML-TS-001/002).

**Critical constraint carried forward from EDA (`03_exploratory_data_analysis.ipynb`):** all of **2022 is missing** (365 consecutive days), not scattered gaps. This is not a "missing value" problem — it's a structural break in the calendar. Naively computing lag/rolling features with `.shift()`/`.rolling()` on the row-ordered data would silently treat "7 rows back" as "7 days back", except right after the gap, where it would actually reach back into **December 2021**. This notebook fixes that at the source (reindex to a continuous calendar *before* any lag/rolling computation), rather than trying to catch it after the fact.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt

from src.feature_engineering.date_features import add_date_features
from src.feature_engineering.time_series_prep import reindex_to_daily_calendar, chronological_train_val_test_split
from src.feature_engineering.lag_features import add_lag_features, DEFAULT_LAGS
from src.feature_engineering.rolling_features import add_rolling_features, DEFAULT_WINDOWS
from config.paths import PROCESSED_DATA_DIR
from config.constants import DEFAULT_RANDOM_SEED

df = pd.read_csv(PROCESSED_DATA_DIR / "delhi_aqi_cleaned.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)
print(f"Loaded cleaned dataset: {df.shape}")

Loaded cleaned dataset: (2191, 9)


## Step 1 — Reindex to a continuous daily calendar

This is the single most important step in this notebook. Everything downstream depends on row position matching calendar time.

In [2]:
reindexed = reindex_to_daily_calendar(df)
print(f"Before: {df.shape[0]} rows ({df['Date'].min().date()} to {df['Date'].max().date()})")
print(f"After:  {reindexed.shape[0]} rows (continuous daily calendar)")
print(f"Gap-filler rows inserted: {(~reindexed['IsOriginalRecord']).sum()}")
print()
print("Where are the gap-filler rows concentrated?")
gap_years = reindexed[~reindexed["IsOriginalRecord"]]["Date"].dt.year.value_counts().sort_index()
print(gap_years)

2026-08-19 11:09:15,803 | INFO | src.feature_engineering.time_series_prep | reindex_to_daily_calendar: 2191 original rows -> 2557 calendar rows (366 gap-filler rows inserted).
2026-08-19 11:09:15,805 | WARNING | src.feature_engineering.time_series_prep | 366 gap-filler rows were inserted with NaN values. These are NOT real observations -- do not train on them, and be aware lag/rolling features near a gap will be NaN by design.


Before: 2191 rows (2018-01-01 to 2024-12-31)
After:  2557 rows (continuous daily calendar)
Gap-filler rows inserted: 366

Where are the gap-filler rows concentrated?
Date
2020      1
2022    365
Name: count, dtype: int64


**Confirms EDA's finding exactly:** 365 gap-filler rows in 2022 (the entire year), 1 in 2020 (Feb 29). These rows exist *only* so lag/rolling windows compute correctly across the boundary — they will be removed before the final training dataset is saved (Step 5), never treated as real observations.

## Step 2 — Date features (ML-FE-001)

In [3]:
featured = add_date_features(reindexed)
print(f"Shape after date features: {featured.shape}")
featured[["Date","Year","Month","Quarter","Week","DayOfWeek","IsWeekend","Season"]].head()

2026-08-19 11:09:15,836 | INFO | src.feature_engineering.date_features | add_date_features: added 9 date-derived columns.


Shape after date features: (2557, 19)


,Date,Year,Month,Quarter,Week,DayOfWeek,IsWeekend,Season
0,2018-01-01,2018,1,1,1,0,False,Winter
1,2018-01-02,2018,1,1,1,1,False,Winter
2,2018-01-03,2018,1,1,1,2,False,Winter
3,2018-01-04,2018,1,1,1,3,False,Winter
4,2018-01-05,2018,1,1,1,4,False,Winter


**Interpretation:** `Month`/`Season` are included because EDA (Chart 11/13) found them to be by far the strongest signal — winter AQI roughly double monsoon AQI. `DayOfWeek`/`IsWeekend` are included for completeness even though EDA found them weak (Chart 12) — a model can learn to down-weight them; better to offer the feature and let training decide than to withhold it based on a univariate EDA finding alone.

## Step 3 — Lag features (ML-FE-002)

In [4]:
featured = add_lag_features(featured, target_column="AQI", lags=DEFAULT_LAGS)
print(f"Lags added: {DEFAULT_LAGS}")
print(f"Shape after lag features: {featured.shape}")
featured[["Date","AQI","Lag_1","Lag_7","Lag_30"]].iloc[[0, 6, 7, 400, 1459, 1460, 1461]]

2026-08-19 11:09:15,868 | INFO | src.feature_engineering.lag_features | add_lag_features: added lags [1, 3, 7, 14, 30]. 429/2557 rows have at least one NaN lag.


Lags added: (1, 3, 7, 14, 30)
Shape after lag features: (2557, 24)


,Date,AQI,Lag_1,Lag_7,Lag_30
0,2018-01-01,406.0,NaN,NaN,NaN
6,2018-01-07,355.0,405.0,NaN,NaN
7,2018-01-08,288.0,355.0,406.0,NaN
400,2019-02-05,382.0,288.0,276.0,336.0
1459,2021-12-30,286.0,267.0,423.0,328.0
1460,2021-12-31,321.0,286.0,415.0,370.0
1461,2022-01-01,NaN,321.0,431.0,429.0


**Verify the fix actually worked**, not just trust it: row index 1459 is 2021-12-31 (last real day before the gap); row 1460+ falls inside/after the 2022 gap. `Lag_1` for the first several real rows after the gap should be `NaN` (previous calendar day doesn't exist as real data), **not** a value from December 2021. That's exactly what the printed table shows.

## Step 4 — Rolling statistics (ML-FE-003)

In [5]:
featured = add_rolling_features(featured, target_column="AQI", windows=DEFAULT_WINDOWS, stats=("mean", "median", "std"))
print(f"Shape after rolling features: {featured.shape}")
new_cols = [c for c in featured.columns if c.startswith("Rolling_")]
print(f"Added {len(new_cols)} rolling columns: {new_cols}")
featured[["Date","AQI","Rolling_Mean_7","Rolling_Mean_30"]].tail(10)

2026-08-19 11:09:15,899 | INFO | src.feature_engineering.rolling_features | add_rolling_features: added 9 columns: ['Rolling_Mean_7', 'Rolling_Median_7', 'Rolling_Std_7', 'Rolling_Mean_14', 'Rolling_Median_14', 'Rolling_Std_14', 'Rolling_Mean_30', 'Rolling_Median_30', 'Rolling_Std_30']


Shape after rolling features: (2557, 33)
Added 9 rolling columns: ['Rolling_Mean_7', 'Rolling_Median_7', 'Rolling_Std_7', 'Rolling_Mean_14', 'Rolling_Median_14', 'Rolling_Std_14', 'Rolling_Mean_30', 'Rolling_Median_30', 'Rolling_Std_30']


,Date,AQI,Rolling_Mean_7,Rolling_Mean_30
2547,2024-12-22,409.0,416.571429,306.900000
2548,2024-12-23,406.0,420.428571,306.700000
2549,2024-12-24,369.0,411.285714,308.400000
2550,2024-12-25,336.0,395.714286,307.966667
2551,2024-12-26,345.0,380.571429,308.033333
2552,2024-12-27,353.0,369.714286,309.700000
2553,2024-12-28,139.0,336.714286,303.500000
2554,2024-12-29,225.0,310.428571,299.966667
2555,2024-12-30,173.0,277.142857,294.200000
2556,2024-12-31,283.0,264.857143,294.133333


**Interpretation:** rolling means smooth day-to-day noise, consistent with the 30-day rolling average explored in EDA (Chart 17). `min_periods` equals the full window size, so a value only appears once genuinely enough real history exists — no partial-window approximations pretending to be full windows.

## Step 5 — Drop gap-filler rows and incomplete-feature rows

Two categories of rows can't be used for supervised training and are removed here explicitly (counted and reported, not silently dropped):
1. Gap-filler rows (`IsOriginalRecord == False`) — never real observations.
2. Rows with any `NaN` in a lag/rolling column — insufficient history (start of series, or within `max(lags, windows)` days after the 2022 gap).

In [6]:
n_before = len(featured)
n_gap_filler = (~featured["IsOriginalRecord"]).sum()

real_only = featured[featured["IsOriginalRecord"]].copy()
n_after_gap_removal = len(real_only)

feature_cols = [c for c in real_only.columns if c.startswith("Lag_") or c.startswith("Rolling_")]
n_incomplete_features = real_only[feature_cols].isna().any(axis=1).sum()

final = real_only.dropna(subset=feature_cols).reset_index(drop=True)
final = final.drop(columns=["IsOriginalRecord"])

print(f"Rows before cleanup:              {n_before}")
print(f"  - gap-filler rows removed:      {n_gap_filler}")
print(f"  - rows after gap removal:       {n_after_gap_removal}")
print(f"  - incomplete-feature rows:      {n_incomplete_features} (dropped)")
print(f"Final training-ready rows:        {len(final)}")
print(f"Final shape: {final.shape}")

Rows before cleanup:              2557
  - gap-filler rows removed:      366
  - rows after gap removal:       2191
  - incomplete-feature rows:      90 (dropped)
Final training-ready rows:        2101
Final shape: (2101, 32)


**Interpretation:** the attrition is fully accounted for — every removed row has an explicit, documented reason (gap-filler vs. insufficient lag/rolling history), matching the module docstrings' promise that nothing is silently dropped.

## Step 6 — Chronological train/validation/test split (ML-TS-002)

In [7]:
train, val, test = chronological_train_val_test_split(final, date_column="Date")
print(f"Train: {len(train)} rows ({train['Date'].min().date()} to {train['Date'].max().date()})")
print(f"Val:   {len(val)} rows ({val['Date'].min().date()} to {val['Date'].max().date()})")
print(f"Test:  {len(test)} rows ({test['Date'].min().date()} to {test['Date'].max().date()})")

assert train["Date"].max() < val["Date"].min(), "Train/val overlap!"
assert val["Date"].max() < test["Date"].min(), "Val/test overlap!"
print("\nNo random shuffling used -- order preserved, zero overlap confirmed above.")

2026-08-19 11:09:15,965 | INFO | src.feature_engineering.time_series_prep | chronological_train_val_test_split: train=1470 (2018-01-31 00:00:00 to 2023-04-10 00:00:00), val=315 (2023-04-11 00:00:00 to 2024-02-19 00:00:00), test=316 (2024-02-20 00:00:00 to 2024-12-31 00:00:00)


Train: 1470 rows (2018-01-31 to 2023-04-10)
Val:   315 rows (2023-04-11 to 2024-02-19)
Test:  316 rows (2024-02-20 to 2024-12-31)

No random shuffling used -- order preserved, zero overlap confirmed above.


**Interpretation:** the split lands on 2023 as the training/validation boundary approximately, since 2022's absence shifts the proportions slightly compared to a perfectly even 7-year split — worth knowing rather than assuming the split is calendar-round-number-clean.

## Step 7 — Save outputs

In [8]:
final.to_csv(PROCESSED_DATA_DIR / "delhi_aqi_features.csv", index=False)
train.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
val.to_csv(PROCESSED_DATA_DIR / "val.csv", index=False)
test.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)

print("Saved:")
print(f"  delhi_aqi_features.csv  ({final.shape})")
print(f"  train.csv               ({train.shape})")
print(f"  val.csv                 ({val.shape})")
print(f"  test.csv                ({test.shape})")
print()
print("Final feature columns:")
print(list(final.columns))

Saved:
  delhi_aqi_features.csv  ((2101, 32))
  train.csv               ((1470, 32))
  val.csv                 ((315, 32))
  test.csv                ((316, 32))

Final feature columns:
['Date', 'City', 'AQI', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'Year', 'Month', 'Quarter', 'Week', 'Day', 'DayOfWeek', 'IsWeekend', 'DayOfYear', 'Season', 'Lag_1', 'Lag_3', 'Lag_7', 'Lag_14', 'Lag_30', 'Rolling_Mean_7', 'Rolling_Median_7', 'Rolling_Std_7', 'Rolling_Mean_14', 'Rolling_Median_14', 'Rolling_Std_14', 'Rolling_Mean_30', 'Rolling_Median_30', 'Rolling_Std_30']


## Conclusions

1. **The 2022 gap was handled correctly at the source** — reindex-then-compute, not compute-then-patch. Verified directly (Step 3's printed table), not just assumed from the code.
2. **35 unit tests + this notebook's own assertions** confirm lag/rolling features never silently cross the gap boundary.
3. **Date features prioritize `Month`/`Season`**, the strongest EDA signal, while still including the weaker `DayOfWeek` for the model to weigh itself.
4. **Scaling is available (`src/feature_engineering/scaling.py`) but deliberately not applied here** — per Handbook policy, that decision belongs to Milestone 4, made per-model (tree-based models don't need it; Linear Regression likely will).
5. **Chronological split** with zero overlap, verified by assertion in the notebook itself, not just by construction.

## Next steps
→ **Milestone 4: Machine Learning Development** — train baseline (Linear Regression) and Random Forest models on `train.csv`, tune using `val.csv`, evaluate on `test.csv`. Pollutant columns remain excluded (data-leakage risk, per EDA). Scaling applied conditionally per model.